## 1. Kurulum ve Repo Klonlama

In [ ]:
!rm -rf /content/QCNN-for-skin-cancer
!git clone https://github.com/irembuseozkose/QCNN-for-skin-cancer.git
%cd QCNN-for-skin-cancer
import sys
sys.path.append("/content/QCNN-for-skin-cancer")

## 2. Bağımlılıklar

In [ ]:
!pip install -q qiskit qiskit-aer scikit-learn matplotlib pandas pillow pylatexenc

## 3. Google Drive Bağlantısı

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Veri Yolları

>  klasörü;  çalıştırıldıktan sonra Drive'a yüklenmiş olmalıdır.

In [ ]:
from pathlib import Path

# Preprocessed .npz dosyalarının bulunduğu klasör
DATA_PATH = Path("/content/drive/MyDrive/Projeler/SkinCancer/features_q_amplitude")

# Checkpoint ve history kayıt klasörü
OUTPUT_DIR = Path("/content/drive/MyDrive/Projeler/SkinCancer/outputs")

print("DATA_PATH :", DATA_PATH)
print("Exists    :", DATA_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

## 5. Import'lar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json

from src.qcnn.encoding import EncodingConfig, build_encoding_circuit
from src.qcnn.ansatz  import build_parametric_qcnn_8q
from src.qcnn.model   import QCNNModel, QCNNModelConfig
from src.qcnn.train   import TrainConfig, fit

## 6. Veri Yükleme

Preprocessing zaten L2-normalize uyguladığından **notebook'ta tekrar normalize etmeye gerek yok**.

In [ ]:
train_npz = np.load(DATA_PATH / "train.npz")
val_npz   = np.load(DATA_PATH / "val.npz")
test_npz  = np.load(DATA_PATH / "test.npz")

x_train, y_train = train_npz["x"].astype(np.float64), train_npz["y"]
x_val,   y_val   = val_npz["x"].astype(np.float64),   val_npz["y"]
x_test,  y_test  = test_npz["x"].astype(np.float64),  test_npz["y"]

with open(DATA_PATH / "metadata.json") as f:
    metadata = json.load(f)

print(f"Train : {x_train.shape}  labels: {y_train.shape}")
print(f"Val   : {x_val.shape}  labels: {y_val.shape}")
print(f"Test  : {x_test.shape}  labels: {y_test.shape}")
print(f"Siniflar ({metadata['num_classes']}): {metadata['label_map']}")

## 7. Veri Doğrulama

In [ ]:
# Temel doğrulama kontrolleri
assert x_train.ndim == 2, "x_train 2D olmali"
assert x_train.shape[1] == 256, f"8 qubit amplitude icin 256 feature bekleniyor, got {x_train.shape[1]}"
assert np.isclose(np.linalg.norm(x_train[0]), 1.0, atol=1e-5), "L2-normalize basarisiz"

print("Unique labels (train):", np.unique(y_train))
print("Sinif dagilimi (train):", {int(k): int(v) for k, v in zip(*np.unique(y_train, return_counts=True))})
print("Ilk ornek norm      :", np.linalg.norm(x_train[0]))
print("Tum kontroller gecti")

## 8. Encoding Devresi (Örnek Görselleştirme)

In [ ]:
encoding_cfg = EncodingConfig(
    n_qubits=8,
    encoding_mode="amplitude",
    rotation_gate="ry",
    add_barriers=False,
)

x0, y0 = x_train[0], y_train[0]
enc_circuit = build_encoding_circuit(x0, encoding_cfg)

print("Ornek etiketi:", y0)
print("Qubit sayisi :", enc_circuit.num_qubits)
enc_circuit.draw("mpl")

## 9. QCNN Ansatz (Güncellenmiş Parametre Sayıları)

In [ ]:
qcnn_circuit, qcnn_params, final_qubits = build_parametric_qcnn_8q(add_barriers=True)

print("Final active qubits:", final_qubits)
for name, vec in qcnn_params.items():
    print(f"  {name}: {len(vec)} params")
print("Toplam QCNN param :", sum(len(v) for v in qcnn_params.values()))

qcnn_circuit.draw("mpl")

## 10. Model Özeti

In [ ]:
qcnn_model = QCNNModel(encoding_cfg=encoding_cfg)
print(qcnn_model.summary())

# Rastgele parametre ile forward pass testi
theta_test = np.random.uniform(-0.1, 0.1, size=qcnn_model.n_trainable_params)
probs_test = qcnn_model.predict_probabilities_statevector(x0, theta_test)
print("Quantum output shape:", probs_test.shape)
print("Quantum output      :", probs_test)
print("Sum                 :", probs_test.sum())

## 11. Eğitim Konfigürasyonu

In [ ]:
train_cfg = TrainConfig(
    data_dir=DATA_PATH,

    n_qubits=8,
    encoding_mode="amplitude",
    rotation_gate="ry",
    add_barriers=False,

    n_classes=9,

    epochs=50,            # SPSA icin yeterli iterasyon
    batch_size=16,        # SPSA gurultusunu azaltir
    seed=42,

    # SPSA decay parametreleri (Spall 1992 optimal)
    lr=0.15,
    c=0.10,
    spsa_alpha=0.602,
    spsa_gamma=0.101,
    spsa_A=10.0,

    head_init_scale=1e-2,
    use_class_weights=True,   # HAM-10000 dengesizligi icin

    max_train_samples=None,
    max_val_samples=None,

    checkpoint_dir=OUTPUT_DIR / "checkpoints",
    history_path=OUTPUT_DIR / "train_history_amplitude.json",
)

## 12. Eğitim

> Colab GPU/CPU'ya göre süre değişir. Her epoch sonunda val_loss ve val_acc yazdırılır.

In [ ]:
model, best_params, history = fit(train_cfg)

## 13. Eğitim Geçmişi

In [ ]:
epochs_df = pd.DataFrame(history["epochs"])
epochs_df.tail(10)

### Loss Eğrisi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_df["epoch"], epochs_df["train_loss"], label="train")
axes[0].plot(epochs_df["epoch"], epochs_df["val_loss"],   label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(epochs_df["epoch"], epochs_df["train_accuracy"], label="train")
axes[1].plot(epochs_df["epoch"], epochs_df["val_accuracy"],   label="val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## 14. Test Değerlendirmesi

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = np.array([model.predict_class(x, best_params) for x in x_test])

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred,
      target_names=list(metadata["label_map"].keys()), zero_division=0))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(cm, display_labels=list(metadata["label_map"].keys()))
disp.plot(ax=ax, xticks_rotation=45, colorbar=True)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

## 15. En İyi Parametreleri Kaydet

In [ ]:
# Zaten train() icinde kaydediliyor; buradan manuel de kaydedebilirsiniz
import numpy as np
save_path = OUTPUT_DIR / "best_params_manual.npy"
np.save(save_path, best_params)
print("Kaydedildi:", save_path)